# Explore the raw data before building the interim dataset

Scratch notebook (nothing saved). Questions to answer before writing the build script:

1. What does the Country lookup contain, and are there special entries that aren't real countries?
2. Which `SuppressionIndex` values occur, and what do suppressed rows look like?
3. How common are rows with a value but no recorded mass?
4. Which records appear under more than one mineral (shared codes)?
5. How are rows spread across flow types and months?

In [ ]:
import json
import tomllib
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO_ROOT / "data" / "raw"

def load(filename):
    """Load a raw JSON file (a list of records) into a DataFrame."""
    with open(RAW / filename, encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

with open(REPO_ROOT / "config" / "minerals.toml", "rb") as f:
    minerals = tomllib.load(f)["minerals"]

In [ ]:
# All 38 OTS files in one table, tagged with the mineral each file belongs to
ots = pd.concat([load(f"hmrc_ots_{key}.json").assign(mineral=key) for key in minerals],
                ignore_index=True)
print(ots.shape)
print("months:", ots["MonthId"].min(), "to", ots["MonthId"].max())
ots.head()

## 1. Country lookup

In [ ]:
country = load("hmrc_country_lookup.json")
print(country.shape)
print(country.columns.tolist())
country.head(10)

In [ ]:
# CountryIds in the trade data that the lookup doesn't know
# (assumes the lookup's ID column is called CountryId: check the column list above)
missing = set(ots["CountryId"]) - set(country["CountryId"])
print("CountryIds missing from lookup:", sorted(missing))

# Entries that look like special codes rather than countries
pattern = "unknown|not known|stores|provisions|confidential|not specified|miscellaneous|high seas"
special = country[country.astype(str).apply(lambda col: col.str.contains(pattern, case=False)).any(axis=1)]
special

In [ ]:
# How many trade rows, and how much value, go to those special entries?
ots[ots["CountryId"].isin(special["CountryId"])].groupby("CountryId")["Value"].agg(["size", "sum"])

## 2. Suppression

In [ ]:
# Which suppression values occur, and how often?
ots["SuppressionIndex"].value_counts()

In [ ]:
# Value and mass for suppressed rows (True) compared with unsuppressed rows (False)
ots.groupby(ots["SuppressionIndex"] > 0)[["Value", "NetMass"]].describe().T

## 3. Value without mass

In [ ]:
# Share of each mineral's rows with a value but no recorded mass
no_mass = (ots["Value"] > 0) & (ots["NetMass"] == 0)
no_mass.groupby(ots["mineral"]).mean().sort_values(ascending=False).round(3)

## 4. Shared codes and duplicates

In [ ]:
record_cols = [c for c in ots.columns if c != "mineral"]

# Exact duplicate records within the same mineral (should be 0)
print("duplicates within a mineral:", ots.duplicated(subset=record_cols + ["mineral"]).sum())

# Records appearing under more than one mineral, by commodity code
shared = ots[ots.duplicated(subset=record_cols, keep=False)]
shared.groupby("CommodityId")["mineral"].unique()

## 5. Flows and months

In [ ]:
# Rows by flow type (1 EU imports, 2 EU exports, 3 non-EU imports, 4 non-EU exports)
ots["FlowTypeId"].value_counts().sort_index()

In [ ]:
# Rows per month: look for gaps, or a recent month that is still incomplete
ots["MonthId"].value_counts().sort_index()

## 6. Follow-up checks

In [ ]:
# 1. Trade rows with the UK itself as partner country (CountryId 6)
uk = ots[ots["CountryId"] == 6]
print(len(uk), "rows, value", f"{uk['Value'].sum():,.0f}")
uk.groupby(["mineral", "FlowTypeId"])["Value"].agg(["size", "sum"])

In [ ]:
# 2. Are the suppressed rows exactly the 'Confidential Country' rows?
suppressed = ots["SuppressionIndex"] == 2
confidential = ots["CountryId"] == 977
print("same rows:", suppressed.eq(confidential).all())

In [ ]:
# 3. Where is the confidential trade concentrated?
# FlowTypeId: 1 EU imports, 2 EU exports, 3 non-EU imports, 4 non-EU exports
conf = ots[confidential].groupby(["mineral", "FlowTypeId"])["Value"].agg(["size", "sum"])
flow_totals = ots.groupby(["mineral", "FlowTypeId"])["Value"].sum()
conf["share_of_flow_value"] = (conf["sum"] / flow_totals.reindex(conf.index)).round(3)
conf.sort_values("sum", ascending=False)

In [ ]:
# 5. Share of each mineral's value (not rows) with no recorded mass
no_mass = (ots["Value"] > 0) & (ots["NetMass"] == 0)
value_no_mass = ots["Value"].where(no_mass, 0).groupby(ots["mineral"]).sum() / ots.groupby("mineral")["Value"].sum()
value_no_mass.sort_values(ascending=False).round(3)

In [ ]:
# 6. Do value-only rows carry a supplementary quantity instead? Share with SuppUnit > 0, by mineral
has_supp = ots.loc[no_mass, "SuppUnit"] > 0
has_supp.groupby(ots.loc[no_mass, "mineral"]).mean().sort_values(ascending=False).round(3)

## 7. HMRC suppressions on our codes

From HMRC's list of currently applied suppressions (checked 16 October 2026). Level 1: nothing published;
Level 2: only totals published, partner countries and ports withheld.

In [ ]:
FLOW_IDS = {"EU Imp": 1, "EU Exp": 2, "Non-EU Imp": 3, "Non-EU Exp": 4}

# (code, mineral, flows, level), transcribed from the current suppressions list
suppression_list = [
    (28170000, "zinc", ["EU Imp", "Non-EU Imp"], 2),
    (28444310, "uranium", ["EU Exp", "Non-EU Exp"], 2),
    (28191000, "chromium", ["EU Imp", "Non-EU Imp"], 1),
    (28199010, "chromium", ["EU Imp", "Non-EU Imp"], 1),
    (28199090, "chromium", ["EU Imp", "Non-EU Imp"], 1),
    (81122110, "chromium", ["EU Exp", "Non-EU Exp"], 1),
    (81122190, "chromium", ["EU Exp", "Non-EU Exp"], 1),
    (28151100, "sodium", ["EU Exp", "Non-EU Exp"], 1),
    (28443019, "uranium", ["EU Exp", "Non-EU Exp"], 1),
    (74111010, "copper", ["EU Exp", "Non-EU Exp"], 1),
    (74111090, "copper", ["EU Exp", "Non-EU Exp"], 1),
]
suppressed_flows = pd.DataFrame(
    [(code, mineral, FLOW_IDS[flow], level) for code, mineral, flows, level in suppression_list for flow in flows],
    columns=["CommodityId", "mineral", "FlowTypeId", "level"],
)
suppressed_flows

In [ ]:
# Which codes and flows do the confidential rows belong to? Expect only the Level 2 codes and flows above
ots[confidential].groupby(["mineral", "CommodityId", "FlowTypeId"]).size()

In [ ]:
# For each suppressed code and flow: rows in our data, how many are confidential, and the months covered.
# Expected: Level 1 has no rows; Level 2 rows are all confidential. Rows that break this pattern
# suggest the suppression started (or was lifted) after January 2022.
def coverage(row):
    sel = ots[(ots["CommodityId"] == row.CommodityId) & (ots["FlowTypeId"] == row.FlowTypeId)]
    return pd.Series({
        "rows": len(sel),
        "confidential_rows": int((sel["CountryId"] == 977).sum()),
        "first_month": sel["MonthId"].min() if len(sel) else None,
        "last_month": sel["MonthId"].max() if len(sel) else None,
    })

suppressed_flows.join(suppressed_flows.apply(coverage, axis=1))